In [ ]:
#IMPORTS
from fileinput import filename
import pandas as pd
import hvplot.pandas
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from st_aggrid import GridOptionsBuilder, AgGrid, GridUpdateMode, DataReturnMode
import holoviews as hv
import numpy as np
import sys
import os
from pathlib import Path, WindowsPath, PureWindowsPath
sys.path.append(str(Path.cwd().parent))
from Intelligence import Utils, cloud_data_collector, DataFramer, Helper
import matplotlib, matplotlib_inline
#%matplotlib inline
from AIML.FinAIML_LinReg import FinAIML_LinReg
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [ ]:
## Welcome to Company Predictive Analytics
def GetSelectedTicker():
        ticker = 'NVDA'
        return ticker
    
ticker = GetSelectedTicker()
tickerdetails = None
TotalProfitLoss = 0


In [ ]:
abspath = os.path.abspath("")
#print(abspath)
tickerdatapath1 = Path(abspath).parent
#print(tickerdatapath1)
location = tickerdatapath1 / 'TickerData' # type: ignore
#print(location)
dfTicker:pd.DataFrame = Utils.GetLocalTickerData(ticker, location)
#print(f'Loading ticker data for ticker {ticker}')
#print(dfTicker)
if(dfTicker is None):
    dfTicker = cloud_data_collector.GetYFinanceData(ticker, '5y')
    if(dfTicker is not None):
        dfTicker.drop('Adj Close', axis=1, inplace=True)
        Utils.SaveTickerData(ticker, dfTicker, location)

if(dfTicker is None):
    print(f"Error loading data for ticker {ticker}.")             
else:
    print(f'Loading ticker data for ticker {ticker}')
    print(dfTicker.head())
    print(dfTicker.tail())


In [ ]:
dfPlot = dfTicker.hvplot.line(
    height=600,
    rot=90,
    responsive=True
)
hv.render(dfPlot, backend='bokeh')

In [ ]:
#Check for Null Values
dataHasNulls = dfTicker.isnull().values.any()
print(f'Null Value Present: {dataHasNulls}')
if(dataHasNulls):
    # Drop missing value
    dfTicker.fillna(value=0, inplace=True)
print(f'Null Value Present: {dfTicker.isnull().values.any()}')


In [ ]:
#Get the Difference between today and previous day
dfTicker['Return'] = dfTicker.Close.diff(periods=1)
dfTicker['Return'].fillna(0, inplace=True)
# Calculate the daily returns using the closing prices and the pct_change function
dfTicker["PercentReturn"] = dfTicker["Close"].pct_change()
dfTicker['PercentReturn'].fillna(0, inplace=True)

print(dfTicker.head())
print(dfTicker.tail())


In [ ]:
dfTicker['DailyTrend'] = dfTicker['Return'].apply(Helper.CalcTrend)
dfTicker['DailyTrend'].fillna(0, inplace=True)
dfTicker.DailyTrend = dfTicker.DailyTrend.astype('int')
dfTicker['Trend'] = 0
dfTicker.Trend = dfTicker.Trend.astype('int')
print(dfTicker.head())
print(dfTicker.tail())


In [ ]:
# Using multiple apply arguments
prevdailytrend = 0
prevtrend = 0
dailytrend = 0
trend = 0

for i in range(len(dfTicker)):
   #print(dfTicker.loc[i, "Name"], dfTicker.loc[i, "Age"])
    #previousrow = dfTicker.iloc[[i-1]]
    dailytrend = dfTicker["DailyTrend"].iloc[i]
    if(i==0): #First Row of the DataSet
        if(dailytrend is np.nan or dailytrend==0):
            dfTicker.loc[dfTicker.index[i], "Trend"] = 0
    else:
        prevdailytrend = dfTicker["DailyTrend"].iloc[i-1]
        prevtrend = dfTicker["Trend"].iloc[i-1]
        trend = 0
        if(prevtrend>=0):                       # type:ignore
            if(dailytrend>=0):                  # type:ignore
                trend = prevtrend + dailytrend  # type:ignore
            elif( dailytrend<0):                # type:ignore
                trend = dailytrend
        elif(prevtrend<0):                      # type:ignore
            if(dailytrend>=0):                  # type:ignore
                trend = dailytrend
            elif( dailytrend<0):                # type:ignore
                trend = prevtrend + dailytrend # type:ignore
        dfTicker.loc[dfTicker.index[i], "Trend"] = trend
print(dfTicker.head())
print(dfTicker.tail())


In [ ]:


# Set the variables for short window and long window periods
sma9 = 9
sma20 = 20
sma50 = 50
sma100 = 100
sma200 = 200
print('Generate the SMAs')
dfTicker["SMA9"] = dfTicker["Close"].rolling(window=sma9).mean()
dfTicker["SMA20"] = dfTicker["Close"].rolling(window=sma20).mean()
dfTicker["SMA50"] = dfTicker["Close"].rolling(window=sma50).mean()
dfTicker["SMA100"] = dfTicker["Close"].rolling(window=sma100).mean()
dfTicker["SMA200"] = dfTicker["Close"].rolling(window=sma200).mean()

dfTicker['SMA9'].fillna(0, inplace=True)
dfTicker['SMA20'].fillna(0, inplace=True)
dfTicker['SMA50'].fillna(0, inplace=True)
dfTicker['SMA100'].fillna(0, inplace=True)
dfTicker['SMA200'].fillna(0, inplace=True)
dfTicker.SMA9 = dfTicker.SMA9.astype('float')
dfTicker.SMA20 = dfTicker.SMA20.astype('float')
dfTicker.SMA50 = dfTicker.SMA50.astype('float')
dfTicker.SMA100 = dfTicker.SMA100.astype('float')
dfTicker.SMA200 = dfTicker.SMA200.astype('float')

print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

In [ ]:
lstDecimalCols = ['Open', 'High', 'Low', 'Close', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']
dfTicker = DataFramer.SetDataFrameColsTo2Decimals(dfData=dfTicker, lstcols=lstDecimalCols)

dfTicker['Open'].fillna(0, inplace=True)
dfTicker['High'].fillna(0, inplace=True)
dfTicker['Low'].fillna(0, inplace=True)
dfTicker['Close'].fillna(0, inplace=True)
dfTicker['SMA9'].fillna(0, inplace=True)
dfTicker['SMA20'].fillna(0, inplace=True)
dfTicker['SMA50'].fillna(0, inplace=True)
dfTicker['SMA100'].fillna(0, inplace=True)
dfTicker['SMA200'].fillna(0, inplace=True)
dfTicker.Open = dfTicker.Open.astype('float')
dfTicker.High = dfTicker.High.astype('float')
dfTicker.Low = dfTicker.Low.astype('float')
dfTicker.Close = dfTicker.Close.astype('float')
dfTicker.SMA9 = dfTicker.SMA9.astype('float')
dfTicker.SMA20 = dfTicker.SMA20.astype('float')
dfTicker.SMA50 = dfTicker.SMA50.astype('float')
dfTicker.SMA100 = dfTicker.SMA100.astype('float')
dfTicker.SMA200 = dfTicker.SMA200.astype('float')

print(dfTicker)
print(f'dfTicker dtypes are {dfTicker.dtypes}')

In [ ]:
print(dfTicker.columns)
featureColumns = ['Open', 'High', 'Low', 'Close', 'Volume', 'PercentReturn', 'Trend', 'SMA9', 'SMA20', 'SMA50', 'SMA100', 'SMA200']

In [ ]:
# Start ML Orchestration
finOrchestrator = FinAIML_LinReg(dfData=dfTicker, featureColumns=featureColumns, targetColName='Close', predictionColName='Predicted', model_type='LinearRegression', train_data_percent=0.5, validation_data_percent=0.3, test_data_percent=0.5, initCapital=0 )

finOrchestrator.Orchestrate()